In [5]:
medicine_keywords = {"Biguanides": ["Metformin"] , 
                     "Sulfonylureas": ["Sulfonylureas", "Gliclazide", "Glipizide", "Glibenclamide"],
                     "Dipeptidyl Peptidase-4 Inhibitors": ["DPP-4 Inhibitors", "Sitagliptin", "Vildagliptin", "Saxagliptin"],
                     "Sodium-Glucose Cotransporter-2 Inhibitors": ["SGLT2 Inhibitors", "Dapagliflozin", "Empagliflozin"],
                     "Glucagon-Like Peptide-1 Agonists": ["GLP-1 Agonists" "Liraglutide", "Exenatide"],
                     "Thiazolidinediones": ["Thiazolidinediones", "Pioglitazone"],
                     "Alpha-glucosidase inhibitors": ["Acarbose"],
                     "Insulin therapy"	: ["Basal", "Prandial", "Biphasic", "Insulin"]}
for m in medicine_keywords:
    print(medicine_keywords[m])

['Metformin']
['Sulfonylureas', 'Gliclazide', 'Glipizide', 'Glibenclamide']
['DPP-4 Inhibitors', 'Sitagliptin', 'Vildagliptin', 'Saxagliptin']
['SGLT2 Inhibitors', 'Dapagliflozin', 'Empagliflozin']
['GLP-1 AgonistsLiraglutide', 'Exenatide']
['Thiazolidinediones', 'Pioglitazone']
['Acarbose']
['Basal', 'Prandial', 'Biphasic', 'Insulin']


In [7]:
import requests
from xml.etree import ElementTree as ET
import time
import json

# Initialize list to hold all results
seen_pmids = set()  # Track unique article IDs
results = []

#email for entrez
email = "...@gmail.com",
tool = "sa_diabetes_literature_scraper"
headers = {'User-Agent': 'sa_diabetes_literature_scraper/1.0 (...@gmail.com)'}


# Define your search terms
search_terms = {"Diabetes": "South Africa AND (diabetes OR blood sugar OR diabetic ketoacidosis OR pancreas OR blood glucose OR diabetic OR hypoglycaemia OR hyperglycaemia OR prediabetes)",
                "Diagnosis": "South Africa AND (glycated haemoglobin OR HbA1c OR A1C)",
                "Management":"South Africa AND (glycaemic control OR exercise for diabetes OR diet for diabetes OR Self-monitoring of blood glucose OR Continuous glucose monitoring)"}

exclusion_clause = "NOT (insulinoma OR short bowel syndrome OR hypopituitarism OR insipidus)" 

# Request wrapper with retries
def safe_request(url, params, headers, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            response.raise_for_status()
            return response
        except requests.exceptions.RequestException as e:
            print(f"Request error: {e} (Attempt {attempt + 1}/{retries})")
            time.sleep(2)
    return None

# Fetching
seen_pmids = set()
results = []

for i in range(10):
    for term_label in search_terms:
        full_query = f"{search_terms[term_label]} {exclusion_clause}"
        search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        search_params = {
            "db": "pubmed",
            "term": full_query,
            "retmax": 200,
            "retmode": "xml",
            "email": email,
            "tool": tool
        }

        search_response = safe_request(search_url, search_params, headers)
        if not search_response:
            print(f"Skipping term '{term_label}' due to repeated connection errors.")
            continue

        root = ET.fromstring(search_response.content)
        pmids = [elem.text for elem in root.findall(".//Id")]
        new_pmids = [pmid for pmid in pmids if pmid not in seen_pmids]
        seen_pmids.update(new_pmids)

        if not new_pmids:
            continue

        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {
            "db": "pubmed",
            "id": ",".join(new_pmids),
            "retmode": "xml",
            "rettype": "abstract",
            "email": email,
            "tool": tool
        }

        fetch_response = safe_request(fetch_url, fetch_params, headers)
        if not fetch_response:
            print(f"Skipping fetch for term '{term_label}' due to connection error.")
            continue

        xml_root = ET.fromstring(fetch_response.content)
        for article in xml_root.findall(".//PubmedArticle"):
            title = article.findtext(".//ArticleTitle")
            abstract_parts = article.findall(".//AbstractText")
            abstract = " ".join([
                part.text.strip() for part in abstract_parts if part is not None and part.text
            ])

            if title and abstract:
                results.append({
                    "search_term": term_label,
                    "title": title.strip(),
                    "abstract": abstract
                })

        # Respect NCBI rate limit
        import random
        time.sleep(random.uniform(1.5, 2.5))

In [ ]:
import os
# Step 4: Save all results to JSON
output_path = r"C:/Users/giftz/OneDrive - University of Pretoria/MIT/MIT 807/data/raw/literature/abstracts/pubmed_abstracts.json"
with open(output_path, "w", encoding="utf-8") as f_json:
    json.dump(results, f_json, ensure_ascii=False, indent=2)

print(f"Results saved to: {output_path}")


Results saved to: C:/Users/giftz/OneDrive - University of Pretoria/MIT/MIT 807/data/raw/literature/abstracts/pubmed_abstracts.json


In [ ]:
len(results)

480